# Machine learning: $R,C$ regression from waveforms

This notebook is split into two parts:

1. **Part 1 — Baseline models** train linear/ridge, random forest, SVR, and MLP with **fixed (hand-picked) hyperparameters**, then summarise test metrics.
2. **Part 2 — Hyperparameter search** applies **`GridSearchCV`** with 3-fold cross-validation to **Random Forest** and **MLP only**. Like Gauss–Newton, this is a **structured search for parameters that minimise a loss** (here, maximise mean $R^2$ over $(R,C)$ on held-out folds); the grid is finite rather than iterative linearisation.

The closing section compares **Gauss–Newton** (physics-based inversion on a test subsample) against ML **predict** times. **Report figures** (`plots/gauss_newton_v3.png`, `plots/ml_v3_pred.png`) are generated in the last cells.

## Part 1 — Setup & evaluation metrics

This notebook loads `data/dataset.pkl` (simulated waveforms $X \in \mathbb{R}^{N \times T \times 4}$ and parameters $y = [R, C])$. Features are flattened to $\mathbb{R}^{N \times 4T}$ and standardized with `StandardScaler` (fit on train only). Train/test split: 80%, 20%.

**Metrics recorded:** training wall time $s$; **R²** on test for $R$ and $C$ separately (1 = perfect); **MAE** in physics units (Ω, F); **MAPE** (%) as a scale-free error proxy (**can look extreme for $R$** when true values are near 1 Ω — interpret next to R²); **architecture** and **complexity** (parameter counts / asymptotic notes).

In [ ]:
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_absolute_percentage_error,
)

DATA_PATH = Path("data/dataset.pkl")
with open(DATA_PATH, "rb") as f:
    bundle = pickle.load(f)

X = np.asarray(bundle["data"])
y = np.asarray(bundle["target"])
n_samples, n_steps, n_channels = X.shape
X_flat = X.reshape(n_samples, n_steps * n_channels)

# Split the dataset into training and testing sets with 80/20 ratio
X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y, test_size=0.2, random_state=42
)

# Scale the feature using sklearn's StandardScaler
x_scaler = StandardScaler()
X_train_s = x_scaler.fit_transform(X_train)
X_test_s = x_scaler.transform(X_test)

rows = []

def record(name, architecture, complexity, train_time, y_true, y_pred):
    r2 = r2_score(y_true, y_pred, multioutput="raw_values")
    mae = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    mape_r = mean_absolute_percentage_error(y_true[:, 0], y_pred[:, 0])
    mape_c = mean_absolute_percentage_error(y_true[:, 1], y_pred[:, 1])
    rows.append(
        {
            "model": name,
            "train_time_s": round(float(train_time), 4),
            "R2_R": round(float(r2[0]), 4),
            "R2_C": round(float(r2[1]), 4),
            "MAE_R_ohm": round(float(mae[0]), 4),
            "MAE_C_F": float(mae[1]),
            "MAPE_R_%": round(100 * float(mape_r), 3),
            "MAPE_C_%": round(100 * float(mape_c), 3),
            "architecture": architecture,
            "complexity_notes": complexity,
        }
    )

# Linear and Ridge Regression

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge

n_feat = X_train_s.shape[1]

lin = LinearRegression()  # Insantiate Linear Reg model from sklearn
t0 = time.perf_counter()
lin.fit(X_train_s, y_train) # Fit the model with the training data 
t_lin = time.perf_counter() - t0
pred = lin.predict(X_test_s)  # Get the predictions for the test set
p_lin = int(lin.coef_.size + lin.intercept_.size)
arch = f"LinearRegression: ŷ = X·Wᵀ + b; W shape {lin.coef_.shape}"
compl = f"{p_lin} fitted parameters; least squares (n={X_train_s.shape[0]}, d={n_feat})"
record("Linear regression", arch, compl, t_lin, y_test, pred)

ridge = Ridge(alpha=1.0, random_state=42) #instatiate Ridge regression model from sklearn 
t0 = time.perf_counter()
ridge.fit(X_train_s, y_train)  # Fit the model with the training data 
t_ridge = time.perf_counter() - t0
pred = ridge.predict(X_test_s)
p_r = int(ridge.coef_.size + ridge.intercept_.size)
arch = f"Ridge(alpha=1.0): same structure; W shape {ridge.coef_.shape}"
compl = f"{p_r} parameters; L2-regularized closed form"
record("Ridge (α=1)", arch, compl, t_ridge, y_test, pred)

# Random Forest Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# One forest per output (R and C) avoids a single multi-output forest over-focusing on one target scale.
base_rf = RandomForestRegressor(
    n_estimators=100,  # Standard number of estimators
    max_depth=None,
    random_state=42,
    n_jobs=-1,
)
rf = MultiOutputRegressor(base_rf) #Instantiate the random forest regressor 
t0 = time.perf_counter()
rf.fit(X_train_s, y_train)  # Fit the model with the training data 
t_rf = time.perf_counter() - t0
pred = rf.predict(X_test_s)
T = rf.estimators_[0].n_estimators
arch = f"MultiOutputRegressor(RandomForestRegressor): {T} trees × 2 outputs (separate forests)"
compl = f"~{2 * T} total trees; training ~O(outputs × T × n log n × d) typical for tree learners"
record("Random forest", arch, compl, t_rf, y_test, pred)

# Support Vector Regression (SVR)

In [5]:
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.compose import TransformedTargetRegressor

# Scale y so R (ohms) does not dominate C (farads); metrics use original units via inverse transform.
svr = TransformedTargetRegressor(
    regressor=MultiOutputRegressor(
        SVR(kernel="rbf", C=10.0, epsilon=0.01, gamma="scale", cache_size=500)
    ),
    transformer=StandardScaler(),
)
t0 = time.perf_counter()
svr.fit(X_train_s, y_train)
t_svr = time.perf_counter() - t0
pred = svr.predict(X_test_s)
mor = svr.regressor_
n_sv = int(sum(int(e.n_support_[0]) for e in mor.estimators_))
arch = "TransformedTargetRegressor(StandardScaler) -> MultiOutputRegressor(SVR(RBF)) per target"
compl = f"~{n_sv} support vectors total; LibSVM ~O(n^2*d) worst case; joint y-scaling helps both outputs"
record("SVR (RBF)", arch, compl, t_svr, y_test, pred)

# MultiLayer Perceptron (MLP)

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor

# Scale R and C jointly so the network is not dominated by the much larger R scale.
mlp = TransformedTargetRegressor(
    regressor=MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation="relu", # Define activation function 
        solver="adam", # Choose optimization algorithm, Adam adapts learning rates per parameter
        max_iter=2000,
        early_stopping=True,
        random_state=42,
        tol=1e-4,
    ),
    transformer=StandardScaler(),
)
t0 = time.perf_counter()
mlp.fit(X_train_s, y_train)  # Fit the model with the training data
t_mlp = time.perf_counter() - t0
pred = mlp.predict(X_test_s)  # Get the predictions for the test set
reg = mlp.regressor_
nparams = int(sum(w.size for w in reg.coefs_) + sum(b.size for b in reg.intercepts_))
arch = (
    f"TransformedTargetRegressor(StandardScaler) → MLP: {n_feat}→{reg.hidden_layer_sizes[0]}→"
    f"{reg.hidden_layer_sizes[1]}→2, {reg.activation}, {reg.solver}"
)
compl = f"{nparams} weights+biases; Adam ~O(epochs·n·params); stopped at iter {reg.n_iter_}"
record("MLP", arch, compl, t_mlp, y_test, pred)

## Part 2 — Hyperparameter search (`GridSearchCV`): Random Forest & MLP

We use **`GridSearchCV`** (scikit-learn) because the hyperparameter spaces are **low-dimensional** and **mostly discrete** (tree counts/depths, MLP widths, $\alpha$, learning rate). That mirrors the **Gauss–Newton** idea of improving a model by **re-solving a sequence of well-defined optimisation subproblems**—here each CV fold fits a candidate on a train split and scores $R^2$ on the validation fold. **`RandomizedSearchCV`** would be preferable for *very* large or continuous spaces; our grids stay small enough for exhaustive search.

**Scoring:** default regressor `r2` = uniform-average $R^2$ across the two targets ($R$ and $C$). **CV:** 3 folds on `X_train_s`, then refit the best estimator on the full training set. **Parallelism:** RF grid uses `n_jobs=-1`; MLP grid uses `n_jobs=1` to avoid oversubscribing workers with neural-net fits.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor

# --- Random Forest: native multi-output (single forest for [R, C]) ---
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [None, 40],
    "min_samples_leaf": [1, 4],
}
rf_gs_est = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_rf = GridSearchCV(  # Define a GridSearchCV object for the hyperparameter tuning of the Random Forest
    rf_gs_est,
    param_grid_rf,
    cv=3,
    scoring="r2",
    refit=True,
    n_jobs=-1,
    verbose=1,
)
t0 = time.perf_counter()
grid_rf.fit(X_train_s, y_train)
t_rf_gs = time.perf_counter() - t0
rf_gs = grid_rf.best_estimator_
pred_rf_gs = rf_gs.predict(X_test_s)
arch_rf = (
    "GridSearchCV(RandomForestRegressor, 3-fold CV, R²); best_params="
    + str(grid_rf.best_params_)
)
_n_rf = int(np.prod([len(v) for v in param_grid_rf.values()]))
compl_rf = (
    f"mean CV R² (best): {grid_rf.best_score_:.4f}; "
    f"{_n_rf} candidates × {grid_rf.n_splits_}-fold CV (~{_n_rf * grid_rf.n_splits_} fits)"
)
record("Random forest (GridSearchCV)", arch_rf, compl_rf, t_rf_gs, y_test, pred_rf_gs)

# --- MLP: keep target scaling (same as Part 1) ---
mlp_gs_pipe = TransformedTargetRegressor(
    regressor=MLPRegressor(
        activation="relu",
        solver="adam",
        max_iter=1200,
        early_stopping=True,
        random_state=42,
        tol=1e-4,
    ),
    transformer=StandardScaler(),
)
# Six MLP architectures × 3 CV folds (early stopping caps epochs); expand the grid only if you have time.
param_grid_mlp = { # Define the hyperparameter grid for the MLPRegressor
    "regressor__hidden_layer_sizes": [(128, 64), (256, 128)],
    "regressor__alpha": [1e-5, 1e-4, 1e-3],
    "regressor__learning_rate_init": [1e-3],
}
grid_mlp = GridSearchCV(
    mlp_gs_pipe,
    param_grid_mlp,
    cv=3,
    scoring="r2",
    refit=True,
    n_jobs=1,
    verbose=1,
)
t0 = time.perf_counter()
grid_mlp.fit(X_train_s, y_train)
t_mlp_gs = time.perf_counter() - t0
mlp_gs = grid_mlp.best_estimator_
pred_mlp_gs = mlp_gs.predict(X_test_s)
reg = mlp_gs.regressor_
nparams = int(sum(w.size for w in reg.coefs_) + sum(b.size for b in reg.intercepts_))
arch_mlp = (
    "GridSearchCV(TransformedTargetRegressor→MLP), 3-fold CV, R²); best_params="
    + str(grid_mlp.best_params_)
)
_n_mlp = int(np.prod([len(v) for v in param_grid_mlp.values()]))
compl_mlp = (
    f"mean CV R² (best): {grid_mlp.best_score_:.4f}; {nparams} weights+biases; "
    f"{_n_mlp} candidates × {grid_mlp.n_splits_}-fold CV; stopped at iter {getattr(reg, 'n_iter_', 'n/a')}"
)
record("MLP (GridSearchCV)", arch_mlp, compl_mlp, t_mlp_gs, y_test, pred_mlp_gs)

print("Best RF params:", grid_rf.best_params_)
print("Best MLP params:", grid_mlp.best_params_)

### Summary table — Part 1 baselines + Part 2 tuned RF/MLP

Run **Part 1** and **Part 2** cells above first. Higher **R²** is better (max 1); lower **MAPE** is better. Training time is wall-clock seconds (Part 2 includes full cross-validation + refit).

In [ ]:
df = pd.DataFrame(rows)
# Plot results of Part 1 and Part 2 
summary = df[
    [
        "model",
        "train_time_s",
        "R2_R",
        "R2_C",
        "MAPE_R_%",
        "MAPE_C_%",
        "MAE_R_ohm",
    ]
].copy()
display(summary)
print("\nFull detail (architecture & complexity):")
display(df[["model", "architecture", "complexity_notes"]])

,model,train_time_s,R2_R,R2_C,MAPE_R_%,MAPE_C_%,MAE_R_ohm
0,Linear regression,0.9672,0.7388,0.9500,120.072,14.680,281.0333
1,Ridge (α=1),0.1013,0.8177,0.9576,116.586,11.400,227.5431
2,Random forest,69.1656,0.9986,0.9696,1.375,7.682,16.7935
3,SVR (RBF),1.9076,0.9635,0.9405,36.328,12.826,100.5911
4,MLP,0.5714,0.8497,0.8344,53.550,33.767,221.8365



Full detail (architecture & complexity):


,model,architecture,complexity_notes
0,Linear regression,"LinearRegression: ŷ = X·Wᵀ + b; W shape (2, 2000)","4002 fitted parameters; least squares (n=1600,..."
1,Ridge (α=1),"Ridge(alpha=1.0): same structure; W shape (2, ...",4002 parameters; L2-regularized closed form
2,Random forest,MultiOutputRegressor(RandomForestRegressor): 1...,~200 total trees; training ~O(outputs × T × n ...
3,SVR (RBF),TransformedTargetRegressor(StandardScaler) -> ...,~2945 support vectors total; LibSVM ~O(n^2*d) ...
4,MLP,TransformedTargetRegressor(StandardScaler) → M...,264514 weights+biases; Adam ~O(epochs·n·params...


### Comparison: classical **Gauss–Newton** (physics-based inversion) vs ML

**`CircuitSimulator.GaussNewton`** refits $R$ and $C$ by minimizing $\sum_t \|x_\mathrm{sim}(t;R,C) - x_\mathrm{meas}(t)\|^2$ using sensitivities $\partial x/\partial R$, $\partial x/\partial C$ — no dataset *training*, but **each test waveform** requires many repeated **Backward Euler + Newton–Raphson** simulations (expensive at inference).

Below we use the **same** random test subsample and simulation settings as in `test.py` / data generation: `amplitude=5`, `f=60` Hz, `delta_t=1e-4`, `T=0.05`, fixed initial guess $R_0{=}1500\,\Omega$, $C_0{=}2.5\,\mu\mathrm{F}$, `max_iter=15`. ML models are **already trained** above (including **Part 2** `GridSearchCV` winners); we only call `.predict` on the masked inputs (cheap).

**Metrics on the subsample:** mean test **MAE** for $R$ (Ω) and $C$ (F); **total wall time** for all GN fits vs one batched predict per model.

If a run hits a **singular Jacobian** in the inner Newton solve, we **retry** Gauss–Newton with initial $(R,C)$ from **Ridge** on that waveform (clipped to the physical ranges) so the comparison still reflects physics-based inversion rather than abandoning hard cases.

In [ ]:
import contextlib
import io

from sklearn.metrics import mean_absolute_error, r2_score

from group_32_circuit_simulator import CircuitSimulator

# Gauss–Newton optimization directly on the circuit simulator, 
# using Ridge regression as a fallback for bad initializations
amplitude = 5.0
f_hz = 60.0
delta_t = 1e-4
T_end = 0.05
x_init = np.zeros((4,))
R0, C0 = 1500.0, 2.5e-6
max_iter_gn = 15

n_test = X_test.shape[0]
X_test_3d = X_test.reshape(n_test, n_steps, n_channels)

rng_cm = np.random.default_rng(43)
n_sub = min(80, n_test)
idx = rng_cm.choice(n_test, size=n_sub, replace=False)
X_sub_s = X_test_s[idx]
y_sub = y_test[idx]

gn_buf = io.StringIO()


def run_gn(obs, R_init, C_init):
    mna = CircuitSimulator(amplitude, f_hz, R_init, C_init)
    with contextlib.redirect_stdout(gn_buf):
        return mna.GaussNewton(
            R_init, C_init, x_init, obs, delta_t, T_end, max_iter=max_iter_gn, noise=False
        )


gn_preds = []
gn_fallbacks = 0
gn_failed = 0
t_gn0 = time.perf_counter()
for k, j in enumerate(idx):
    obs = X_test_3d[j]
    try:
        R_hat, C_hat, _ = run_gn(obs, R0, C0)
    except np.linalg.LinAlgError:
        Ri, Ci = ridge.predict(X_sub_s[k : k + 1])[0]
        Ri = float(np.clip(Ri, 1.0, 2500.0))
        Ci = float(np.clip(Ci, 0.1e-6, 5e-6))
        try:
            R_hat, C_hat, _ = run_gn(obs, Ri, Ci)
            gn_fallbacks += 1
        except np.linalg.LinAlgError:
            R_hat, C_hat = np.nan, np.nan
            gn_failed += 1
    gn_preds.append([R_hat, C_hat])
t_gn = time.perf_counter() - t_gn0
gn_preds = np.asarray(gn_preds)

def subsample_mae(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    return float(mae[0]), float(mae[1])


def subsample_r2(y_true, y_pred):
    r2 = r2_score(y_true, y_pred, multioutput="raw_values")
    return float(r2[0]), float(r2[1])


compare_rows = []
gn_ok = np.isfinite(gn_preds[:, 0]) & np.isfinite(gn_preds[:, 1])
if gn_ok.any():
    m_ae_r, m_ae_c = subsample_mae(y_sub[gn_ok], gn_preds[gn_ok])
    r2r, r2c = subsample_r2(y_sub[gn_ok], gn_preds[gn_ok])
else:
    m_ae_r = m_ae_c = r2r = r2c = float("nan")
compare_rows.append(
    {
        "method": "Gauss–Newton (physics)",
        "role": f"inference only; warm-start fallback: {gn_fallbacks}/{n_sub}, failed: {gn_failed}",
        "wall_time_s": round(t_gn, 4),
        "MAE_R_ohm": round(m_ae_r, 4),
        "MAE_C_F": m_ae_c,
        "R2_R": round(r2r, 4),
        "R2_C": round(r2c, 4),
        "architecture": "Iterative Gauss–Newton on nonlinear MNA + Backward Euler trajectory",
    }
)

models = [
    ("Linear regression", lin),
    ("Ridge (α=1)", ridge),
    ("Random forest", rf),
    ("Random forest (GridSearchCV)", rf_gs),
    ("SVR (RBF)", svr),
    ("MLP", mlp),
    ("MLP (GridSearchCV)", mlp_gs),
]

for name, model in models:
    t0 = time.perf_counter()
    pred_sub = model.predict(X_sub_s)
    t_pred = time.perf_counter() - t0
    m_ae_r, m_ae_c = subsample_mae(y_sub, pred_sub)
    r2r, r2c = subsample_r2(y_sub, pred_sub)
    compare_rows.append(
        {
            "method": name,
            "role": "predict on subsample (training done earlier)",
            "wall_time_s": round(t_pred, 6),
            "MAE_R_ohm": round(m_ae_r, 4),
            "MAE_C_F": m_ae_c,
            "R2_R": round(r2r, 4),
            "R2_C": round(r2c, 4),
            "architecture": "see ML summary rows above",
        }
    )

compare_df = pd.DataFrame(compare_rows)
display(
    compare_df[
        [
            "method",
            "role",
            "wall_time_s",
            "R2_R",
            "R2_C",
            "MAE_R_ohm",
        ]
    ]
)
print(
    f"Gauss–Newton subset: n={n_sub}, primary init R0={R0} Ω, C0={C0} F, max_iter={max_iter_gn}; "
    f"metrics use {int(gn_ok.sum())} converged runs ({gn_failed} failed even after Ridge warm-start)."
)
print(
    "Interpretation: GN matches the simulator (self-consistent physics); ML maps waveforms to parameters without repeatedly simulating the circuit."
)

/Users/louissalanon/ECSE343Project/group_32_circuit_simulator.py:49: RuntimeWarning: overflow encountered in scalar power
  [0, 0, -1/(R**2), 0],
/Users/louissalanon/ECSE343Project/group_32_circuit_simulator.py:49: RuntimeWarning: overflow encountered in scalar power
  [0, 0, -1/(R**2), 0],
/Users/louissalanon/ECSE343Project/group_32_circuit_simulator.py:49: RuntimeWarning: overflow encountered in scalar power
  [0, 0, -1/(R**2), 0],
/Users/louissalanon/ECSE343Project/group_32_circuit_simulator.py:49: RuntimeWarning: overflow encountered in scalar power
  [0, 0, -1/(R**2), 0],
/Users/louissalanon/ECSE343Project/group_32_circuit_simulator.py:49: RuntimeWarning: overflow encountered in scalar power
  [0, 0, -1/(R**2), 0],
/Users/louissalanon/ECSE343Project/group_32_circuit_simulator.py:49: RuntimeWarning: overflow encountered in scalar power
  [0, 0, -1/(R**2), 0],
/Users/louissalanon/ECSE343Project/group_32_circuit_simulator.py:49: RuntimeWarning: overflow encountered in scalar power
  

,method,role,wall_time_s,R2_R,R2_C,MAE_R_ohm
0,Gauss–Newton (physics),"inference only; warm-start fallback: 8/80, fai...",29.117500,0.9983,0.9992,18.7154
1,Linear regression,predict on subsample (training done earlier),0.001655,0.7555,0.8679,274.3069
2,Ridge (α=1),predict on subsample (training done earlier),0.000409,0.8362,0.8720,214.9004
3,Random forest,predict on subsample (training done earlier),0.063747,0.9987,0.9540,17.4725
4,SVR (RBF),predict on subsample (training done earlier),0.393953,0.9528,0.9372,112.9912
5,MLP,predict on subsample (training done earlier),0.001214,0.8141,0.9032,248.0093


Gauss–Newton subset: n=80, primary init R0=1500.0 Ω, C0=2.5e-06 F, max_iter=15; metrics use 76 converged runs (3 failed even after Ridge warm-start).
Interpretation: GN matches the simulator (self-consistent physics); ML maps waveforms to parameters without repeatedly simulating the circuit.


### Report figures — `plots/gauss_newton_v3.png` and `plots/ml_v3_pred.png`

Uses `data/measurements.csv`. **Gauss–Newton** uses the same initial guess as `test.py` / report text ($R_0=2500\,\Omega$, $C_0=3\,\mu\text{F}$). **ML curve** uses the **tuned** random forest (`rf_gs`) to predict $(R,C)$ from the flattened measurement, then forward-simulates $V_3$.

In [1]:
import matplotlib.pyplot as plt

from group_32_circuit_simulator import CircuitSimulator

PLOTS = Path("plots")
PLOTS.mkdir(parents=True, exist_ok=True)
MEAS = Path("data/measurements.csv")
x_meas = np.loadtxt(MEAS, delimiter=",")

amplitude = 5.0
f_hz = 60.0
delta_t = 1e-4
T_end = 0.05
x_init = np.zeros(4)

# --- Gauss–Newton on real measurements (report-style initial guess) ---
R_guess, C_guess = 2500.0, 3e-6
mna_gn = CircuitSimulator(amplitude, f_hz, R_guess, C_guess)
R_gn, C_gn, _cost = mna_gn.GaussNewton(
    R_guess, C_guess, x_init, x_meas, delta_t, T_end, max_iter=15, noise=False
)
mna_gn_sim = CircuitSimulator(amplitude, f_hz, R_gn, C_gn)
x_gn, t_axis = mna_gn_sim.BEuler(x_init, delta_t, T_end, noise=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t_axis, x_meas[:, 2], color="0.25", lw=1.2, alpha=0.65, label="Measured $V_3$")
ax.plot(t_axis, x_gn[:, 2], "b-", lw=1.6, label=f"GN sim. $V_3$ ($R$={R_gn:.0f} $\\Omega$, $C$={C_gn*1e6:.2f} $\\mu$F)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("$V_3$ (V)")
ax.set_title("Gauss–Newton vs measurement ($V_3$)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, ls="--", alpha=0.35)
fig.tight_layout()
fig.savefig(PLOTS / "gauss_newton_v3.png", dpi=200)
plt.close(fig)

# --- Best RF (GridSearchCV): predict R,C then simulate V3 ---
X_meas_flat = x_meas.reshape(1, -1)
X_meas_s = x_scaler.transform(X_meas_flat)
R_rf, C_rf = rf_gs.predict(X_meas_s)[0]
mna_rf = CircuitSimulator(amplitude, f_hz, float(R_rf), float(C_rf))
x_rf, t2 = mna_rf.BEuler(x_init, delta_t, T_end, noise=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t2, x_meas[:, 2], color="0.25", lw=1.2, alpha=0.65, label="Measured $V_3$")
ax.plot(t2, x_rf[:, 2], color="darkorange", lw=1.6, label=f"RF+CV sim. $V_3$ ($R$={R_rf:.0f} $\\Omega$, $C$={C_rf*1e6:.2f} $\\mu$F)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("$V_3$ (V)")
ax.set_title("Tuned random forest vs measurement ($V_3$)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, ls="--", alpha=0.35)
fig.tight_layout()
fig.savefig(PLOTS / "ml_v3_pred.png", dpi=200)
plt.close(fig)

print("Wrote:", PLOTS / "gauss_newton_v3.png")
print("Wrote:", PLOTS / "ml_v3_pred.png")

NameError: name 'Path' is not defined